# 05a — Synthetic Ground Truth (Attribution-First)

**Goal.** Build the Phase-1a evaluation set without an SME. For each
selected source span (1-3 consecutive sentences within a single chunk),
ask a non-OpenAI-family foundation model (synth-GT role) to propose an
auditor-style question + gold answer whose support is the span **by
construction**. The span's `sentence_id`s are the gold citations.

**Invariants.**
- RAG generator, synth-GT author, and judge must be three distinct model
  identities. Enforced at runtime by
  `AzureConfig.assert_three_distinct_models` inside `llm.get_binding`.
- The span's `sentence_id`s become the gold citation set for every answer
  sentence. The `synth-gt-quality-gates` step expands that set with the
  union labeler.

**Output.** `data/ground_truth/synthetic/<run_id>/` with
`items.jsonl`, `failures.jsonl`, and `manifest.json`.

In [ ]:
from pathlib import Path

from sentcite.llm import describe_roles
from sentcite.synth_gt import (
    generate_synth_gt,
    load_corpus_chunks,
    select_spans,
    write_synth_gt_run,
)

describe_roles()

In [ ]:
chunks = load_corpus_chunks('data/chunks')
len(chunks)

In [ ]:
targets = {'easy': 40, 'medium': 35, 'hard': 25}
spans = select_spans(chunks, target_per_difficulty=targets, seed=7)
len(spans), {d: sum(1 for s in spans if s.difficulty == d) for d in ('easy','medium','hard')}

In [ ]:
def _progress(done, total, items, failures):
    if done % 10 == 0 or done == total:
        print(f'[{done:>3}/{total}] items={len(items)} failures={len(failures)}', flush=True)

run = generate_synth_gt(
    spans,
    target_counts=targets,
    on_progress=_progress,
)
print(run.to_manifest())

In [ ]:
paths = write_synth_gt_run(run, out_dir='data/ground_truth/synthetic')
paths

In [ ]:
# Peek at the first 3 items so we have a sanity check that the tone
# is auditor-style and the span is meaningfully supportive.
for it in run.items[:3]:
    print(f"\n[{it.difficulty}] {it.question_id}  ({it.document_id} p{it.page})")
    print(f"  span: {it.source_span_sentence_ids}")
    print(f"  Q: {it.question}")
    print(f"  A: {it.gold_answer}")